<a href="https://colab.research.google.com/github/ThousandAI/PythonAI-010/blob/main/API_Youbike.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""youbike_click_TEACHER.ipynb  ——  【老師版｜完整解答】

專案：YouBike 借還車互動地圖
================================
使用者流程：
  1. 執行後出現全台北地圖 + 「借車 / 還車」切換 + 「清除」按鈕
  2. 滑鼠移到任何站點上 → 直接顯示該站可借幾台、可還幾台（不用點）
  3. 在地圖上點一下任一位置 → 自動標出最近 5 站、拉線、右側列清單

資料來源：YouBike2.0 臺北市公共自行車即時資訊（每分鐘更新，免金鑰）
環境：Google Colab ｜ 地圖：ipyleaflet
欄位：可借=available_rent_bikes、可還=available_return_bikes、
      經緯度=latitude/longitude、總車位=Quantity。
"""

# ============================================================
# 【格 0】安裝套件（裝完建議「執行階段→重新啟動並全部執行」一次）
# ============================================================
get_ipython().system('pip -q install ipyleaflet')

from google.colab import output
output.enable_custom_widget_manager()

# ============================================================
# 【格 1】抓 API + 資料清理
# ============================================================
import requests
import pandas as pd

URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"


def fetch_youbike():
    d = pd.DataFrame(requests.get(URL, timeout=10).json())

    rename = {}
    if "latitude" in d.columns:  rename["latitude"] = "lat"
    if "longitude" in d.columns: rename["longitude"] = "lng"
    for c in ["Quantity", "tot", "total", "totalstationcount"]:
        if c in d.columns:
            rename[c] = "capacity"
            break
    d = d.rename(columns=rename)

    for col in ["available_rent_bikes", "available_return_bikes", "capacity", "lat", "lng"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    d = d.dropna(subset=["lat", "lng"]).reset_index(drop=True)
    return d


df = fetch_youbike()
print("抓取成功，總站數：", len(df))

# ============================================================
# 【格 2】距離公式 + 找最近站（依借/還模式）
# ============================================================
from math import radians, sin, cos, sqrt, atan2

WALK_KMH = 4.8


def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def nearest(lat, lng, mode="借車", top=5):
    field = "available_rent_bikes" if mode == "借車" else "available_return_bikes"
    d = df[df[field] > 0].copy()
    d["dist"] = d.apply(lambda r: haversine(lat, lng, r["lat"], r["lng"]), axis=1)
    d["walk"] = (d["dist"] / WALK_KMH * 60).round(1)
    d["avail"] = d[field]
    return d.sort_values("dist").head(top)[["sna", "sarea", "avail", "dist", "walk", "lat", "lng"]]

# ============================================================
# 【格 3】互動地圖：hover 顯示可借可還 + 點擊跳最近 5 站
# ============================================================
from ipyleaflet import Map, Marker, CircleMarker, Polyline, AwesomeIcon
import ipywidgets as widgets
from IPython.display import display

m = Map(center=(25.0478, 121.5319), zoom=12, scroll_wheel_zoom=True)

# --- 先把所有站點畫成底圖小圓點，滑鼠移上去(hover)顯示可借/可還 ---
for _, r in df.iterrows():
    rent = int(r["available_rent_bikes"]) if pd.notna(r["available_rent_bikes"]) else 0
    ret = int(r["available_return_bikes"]) if pd.notna(r["available_return_bikes"]) else 0
    base = CircleMarker(
        location=(r["lat"], r["lng"]), radius=3,
        color="#3388ff", fill_color="#3388ff", fill_opacity=0.5, weight=1,
        title=f"{r['sna']}｜可借 {rent} 台｜可還 {ret} 台",   # ← 滑鼠移上去顯示
    )
    m.add_layer(base)

# --- 控制列：借/還切換、清除按鈕、結果清單 ---
mode_toggle = widgets.ToggleButtons(options=["借車", "還車"], value="借車", description="模式:")
clear_btn = widgets.Button(description="清除", button_style="warning", icon="eraser")
info_box = widgets.HTML(value="<b>👉 滑鼠移到站點看車數；點地圖任一位置找最近 5 站</b>")

dynamic_layers = []   # 記住點擊後畫的圖層，方便清除


def clear_dynamic(_=None):
    for layer in dynamic_layers:
        try:
            m.remove_layer(layer)
        except Exception:
            pass
    dynamic_layers.clear()
    info_box.value = "<b>已清除，請重新點地圖或滑到站點上查看</b>"


def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lng = kwargs["coordinates"]
    clear_dynamic()

    mode = mode_toggle.value
    result = nearest(lat, lng, mode=mode, top=5)
    kind = "可借" if mode == "借車" else "可還"

    click_marker = Marker(location=(lat, lng),
                          icon=AwesomeIcon(name="user", marker_color="red"))
    m.add_layer(click_marker); dynamic_layers.append(click_marker)

    rows = []
    for i, (_, r) in enumerate(result.iterrows(), 1):
        rent = int(r["avail"])
        cm = CircleMarker(location=(r["lat"], r["lng"]), radius=8,
                          color="green", fill_color="green", fill_opacity=0.9,
                          title=f"{r['sna']}｜{kind} {rent} 台")   # 最近站也可 hover
        m.add_layer(cm); dynamic_layers.append(cm)

        line = Polyline(locations=[(lat, lng), (r["lat"], r["lng"])],
                        color="blue", weight=2, dash_array="5")
        m.add_layer(line); dynamic_layers.append(line)

        rows.append(f"{i}. {r['sna']}（{kind} {rent} 台）"
                    f"｜{r['dist']:.2f} km｜步行 {r['walk']} 分")

    info_box.value = (f"<b>📍 最近、且{kind}的前 5 站（{mode}）</b><br>"
                      + "<br>".join(rows))


m.on_interaction(on_click)
clear_btn.on_click(clear_dynamic)


def on_mode_change(_):
    info_box.value = f"<b>已切到「{mode_toggle.value}」，請重新點一下地圖</b>"
mode_toggle.observe(on_mode_change, names="value")

display(widgets.HBox([mode_toggle, clear_btn]))
display(widgets.HBox([m, info_box]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.4 MB/s eta 0:00:00
抓取成功，總站數： 1774
